# H&M Image-History Full Training Baseline

Bu notebook, Perseptron projesindeki multimodal deneyin **image-history baseline** modelini eğitmek için hazırlanmıştır. Bu model tabular müşteri/ürün metadata'sı kullanmaz. Yalnızca aday ürünün EfficientNet-B0 embedding'i, müşterinin geçmiş görsel tercih profili, cosine similarity ve görsel geçmiş uzunluğu ile eğitilir.

Bu baseline, late fusion modelindeki görsel branch'in tek başına ne kadar bilgi taşıdığını ölçer. Böylece final raporda şu üçlü ablation karşılaştırması tamamlanır: tabular-only, image-history ve multimodal late fusion.


## 1. Kurulum ve Deney Ayarları

Bu hücrede Kaggle input yolları, full embedding dataset path'leri ve streaming training hiperparametreleri tanımlanır. Embedding dosyaları PC'den Kaggle Dataset olarak yüklendiği için notebook tekrar EfficientNet çalıştırmaz; doğrudan hazır embedding matrisini okur.


In [1]:
from __future__ import annotations

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

FAST_RUN = False
RANDOM_SEED = 42
WORK_DIR = Path("/kaggle/working")

STREAM_POSITIVE_CHUNK_SIZE = 200_000
STREAM_EPOCHS = 3
STREAM_BATCH_SIZE = 4096
VALIDATION_POSITIVE_ROWS = 120_000
NEGATIVES_PER_POSITIVE = 2
LEARNING_RATE = 1e-3

MODEL_PATH = WORK_DIR / "image_history_streaming_full.pt"
RESULTS_PATH = WORK_DIR / "image_history_full_results.csv"

EMBEDDINGS_DIR = Path("/kaggle/input/datasets/smoke78/article-image-embeddings-popular-npy")
EMBEDDING_IDS_DIR = Path("/kaggle/input/datasets/smoke78/article-image-embedding-ids-popular-csv")


def log(message: str) -> None:
    print(message, flush=True)


def get_device(stage: str) -> torch.device:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        name = torch.cuda.get_device_name(0)
        allocated = torch.cuda.memory_allocated(0) / 1024**3
        reserved = torch.cuda.memory_reserved(0) / 1024**3
        log(f"[{stage}] CUDA available: True | GPU: {name} | allocated={allocated:.2f}GB | reserved={reserved:.2f}GB")
    else:
        log(f"[{stage}] CUDA available: False | running on CPU")
    return device


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def l2_normalize(values: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    return values / np.maximum(norms, eps)


def find_hm_input_dir() -> Path:
    candidates = []
    for root in [Path("/kaggle/input"), Path("/kaggle/input/competitions")]:
        if not root.exists():
            continue
        for path in root.iterdir():
            if path.is_dir() and (path / "transactions_train.csv").exists():
                candidates.append(path)
            if path.is_dir():
                for child in path.iterdir():
                    if child.is_dir() and (child / "transactions_train.csv").exists():
                        candidates.append(child)
    if not candidates:
        available = "\n".join(str(path) for path in Path("/kaggle/input").iterdir())
        raise FileNotFoundError(f"Could not find H&M input directory. Available entries:\n{available}")
    return candidates[0]


INPUT_DIR = find_hm_input_dir()
log(f"Using input directory: {INPUT_DIR}")

TRANSACTIONS_PATH = INPUT_DIR / "transactions_train.csv"
CUSTOMERS_PATH = INPUT_DIR / "customers.csv"
ARTICLES_PATH = INPUT_DIR / "articles.csv"


Using input directory: /kaggle/input/competitions/h-and-m-personalized-fashion-recommendations


## 2. Veri, Embedding ve Görsel Profil Fonksiyonları

Bu bölümde H&M transaction verisi yüklenir, hazır embedding cache okunur ve her müşteri için geçmiş satın almalardan görsel profil toplamları oluşturulur. Image-history baseline'da tabular kolonlar modele verilmez.


In [2]:
def first_file_in_dir(path: Path, token: str | None = None) -> Path:
    if path.is_file():
        return path
    if not path.exists():
        raise FileNotFoundError(path)
    files = [candidate for candidate in path.iterdir() if candidate.is_file()]
    if token is not None:
        matching = [candidate for candidate in files if token in candidate.name.lower()]
        if matching:
            return matching[0]
    if files:
        return files[0]
    raise FileNotFoundError(f"No files found under {path}")


def load_image_embeddings():
    embeddings_path = first_file_in_dir(EMBEDDINGS_DIR, "embedding")
    ids_path = first_file_in_dir(EMBEDDING_IDS_DIR, "id")

    log(f"Loading embeddings: {embeddings_path}")
    log(f"Loading embedding ids: {ids_path}")

    article_ids = (
        pd.read_csv(ids_path, dtype={"article_id": str})["article_id"]
        .astype(str)
        .str.zfill(10)
        .tolist()
    )
    embeddings = np.load(embeddings_path).astype("float32")

    if len(article_ids) != len(embeddings):
        raise ValueError(f"Embedding/id mismatch: {len(embeddings):,} embeddings vs {len(article_ids):,} ids")

    embeddings = l2_normalize(embeddings)
    article_to_index = {article_id: index for index, article_id in enumerate(article_ids)}
    log(f"Loaded embeddings: {embeddings.shape}")
    return article_ids, embeddings, article_to_index


def reduce_transactions_memory(transactions: pd.DataFrame) -> pd.DataFrame:
    transactions["article_id"] = transactions["article_id"].astype(str).str.zfill(10)
    transactions["price"] = transactions["price"].astype("float32")
    transactions["sales_channel_id"] = transactions["sales_channel_id"].astype("int8")
    return transactions


def load_transactions():
    log("Loading H&M transactions...")
    transactions = pd.read_csv(TRANSACTIONS_PATH, dtype={"article_id": str})
    transactions = reduce_transactions_memory(transactions)
    log(f"Transactions: {len(transactions):,}")
    return transactions


def build_customer_visual_sums(transactions, embeddings, article_to_index):
    log("Filtering transactions to embedded articles...")
    transactions = transactions[transactions["article_id"].isin(article_to_index)].copy()
    log(f"Embedded transactions: {len(transactions):,}")

    customer_ids = sorted(transactions["customer_id"].unique())
    customer_to_index = {customer_id: index for index, customer_id in enumerate(customer_ids)}
    log(f"Customers with visual history: {len(customer_ids):,}")

    profile_sums = np.zeros((len(customer_ids), embeddings.shape[1]), dtype="float32")
    profile_counts = np.zeros(len(customer_ids), dtype="float32")

    for row in tqdm(transactions[["customer_id", "article_id"]].itertuples(index=False), total=len(transactions), desc="Customer visual profiles"):
        customer_index = customer_to_index[row.customer_id]
        article_index = article_to_index[row.article_id]
        profile_sums[customer_index] += embeddings[article_index]
        profile_counts[customer_index] += 1.0

    return transactions, customer_ids, customer_to_index, profile_sums, profile_counts


def make_visual_arrays(data, embeddings, profile_sums, profile_counts, batch_size=50_000):
    article_indices = data["article_index"].to_numpy(dtype="int64")
    customer_indices = data["customer_index"].to_numpy(dtype="int64")
    pair_counts = data["pair_purchase_count"].to_numpy(dtype="float32")

    visual_similarity = np.empty(len(data), dtype="float32")
    visual_history_count = np.empty(len(data), dtype="float32")

    for start in tqdm(range(0, len(data), batch_size), desc="Visual feature batches", leave=False):
        end = min(start + batch_size, len(data))
        aidx = article_indices[start:end]
        cidx = customer_indices[start:end]
        pc = pair_counts[start:end]

        article_emb = embeddings[aidx]
        sums = profile_sums[cidx] - pc[:, None] * article_emb
        counts = profile_counts[cidx] - pc
        profiles = sums / np.maximum(counts, 1.0)[:, None]
        profiles = l2_normalize(profiles)
        profiles[counts <= 0] = 0

        visual_similarity[start:end] = np.sum(profiles * article_emb, axis=1).astype("float32")
        visual_history_count[start:end] = counts.astype("float32")

        del article_emb, sums, profiles, counts
        gc.collect()

    return article_indices, customer_indices, pair_counts, visual_similarity, visual_history_count


## 3. Image-History Model ve Streaming Eğitim Fonksiyonları

Image-history modeli, aday ürün embedding'i ile müşterinin görsel profil embedding'ini birlikte kullanır. Ek olarak cosine similarity ve görsel geçmiş uzunluğu modele verilir. Tabular metadata bu modelde kullanılmaz.


In [3]:
class ImageHistoryDataset(Dataset):
    def __init__(self, article_idx, customer_idx, pair_counts, visual_similarity, visual_history_count, labels):
        self.article_idx = torch.tensor(article_idx, dtype=torch.long)
        self.customer_idx = torch.tensor(customer_idx, dtype=torch.long)
        self.pair_counts = torch.tensor(pair_counts, dtype=torch.float32)
        self.visual_similarity = torch.tensor(visual_similarity, dtype=torch.float32)
        self.visual_history_count = torch.tensor(visual_history_count, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.article_idx[idx],
            self.customer_idx[idx],
            self.pair_counts[idx],
            self.visual_similarity[idx],
            self.visual_history_count[idx],
            self.labels[idx],
        )


class ImageHistoryMLP(nn.Module):
    def __init__(self, image_dim: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(image_dim * 2 + 2, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(256, 1),
        )

    def forward(self, article_emb, profile_emb, visual_similarity, visual_history_count):
        visual_extra = torch.stack(
            [visual_similarity, torch.log1p(torch.clamp(visual_history_count, min=0.0))],
            dim=1,
        )
        x = torch.cat([article_emb, profile_emb, visual_extra], dim=1)
        return self.network(x).squeeze(1)


def prepare_pair_chunk(positive_chunk, chunk_id, article_pool, rng, article_to_index, customer_to_index):
    positives = positive_chunk.copy()
    positives["label"] = 1

    neg_customers = np.repeat(positives["customer_id"].values, NEGATIVES_PER_POSITIVE)
    neg_articles = rng.choice(article_pool, size=len(neg_customers), replace=True)
    negatives = pd.DataFrame({"customer_id": neg_customers, "article_id": neg_articles, "label": 0})

    data = pd.concat([positives, negatives], ignore_index=True)
    data = data.drop_duplicates(["customer_id", "article_id", "label"])
    data = data[data["customer_id"].isin(customer_to_index)].copy()
    data = data.sample(frac=1.0, random_state=RANDOM_SEED + chunk_id).reset_index(drop=True)

    data["article_index"] = data["article_id"].map(article_to_index).astype("int64")
    data["customer_index"] = data["customer_id"].map(customer_to_index).astype("int64")
    data["pair_purchase_count"] = data["label"].astype("float32")

    del positives, negatives, neg_customers, neg_articles
    gc.collect()
    return data


def make_visual_batch(article_idx, customer_idx, pair_counts, article_tensor, profile_sum_tensor, profile_count_tensor):
    article_emb = article_tensor[article_idx]
    sums = profile_sum_tensor[customer_idx] - pair_counts.unsqueeze(1) * article_emb
    counts = profile_count_tensor[customer_idx] - pair_counts
    profiles = sums / torch.clamp(counts, min=1.0).unsqueeze(1)
    profiles = torch.nn.functional.normalize(profiles, p=2, dim=1)
    profiles = torch.where(counts.unsqueeze(1) > 0, profiles, torch.zeros_like(profiles))
    visual_similarity = torch.sum(profiles * article_emb, dim=1)
    return article_emb, profiles, visual_similarity, counts


def build_dataset_from_chunk(data, embeddings, profile_sums, profile_counts):
    article_idx, customer_idx, pair_counts, visual_similarity, visual_history_count = make_visual_arrays(
        data,
        embeddings,
        profile_sums,
        profile_counts,
        batch_size=50_000,
    )
    labels = data["label"].to_numpy(dtype="float32")
    return ImageHistoryDataset(article_idx, customer_idx, pair_counts, visual_similarity, visual_history_count, labels)


def train_one_chunk(model, optimizer, criterion, data, article_tensor, profile_sum_tensor, profile_count_tensor, device):
    dataset = build_dataset_from_chunk(data, embeddings, profile_sums, profile_counts)
    loader = DataLoader(dataset, batch_size=STREAM_BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=(device.type == "cuda"))

    model.train()
    losses = []
    for article_idx, customer_idx, pair_counts, visual_similarity, visual_history_count, labels_b in loader:
        article_idx = article_idx.to(device, non_blocking=True)
        customer_idx = customer_idx.to(device, non_blocking=True)
        pair_counts = pair_counts.to(device, non_blocking=True)
        labels_b = labels_b.to(device, non_blocking=True)

        article_emb, profiles, visual_similarity_b, history_counts = make_visual_batch(
            article_idx,
            customer_idx,
            pair_counts,
            article_tensor,
            profile_sum_tensor,
            profile_count_tensor,
        )

        optimizer.zero_grad()
        logits = model(article_emb, profiles, visual_similarity_b, history_counts)
        loss = criterion(logits, labels_b)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    del dataset, loader
    gc.collect()
    return float(np.mean(losses))


def evaluate_model(model, data, article_tensor, profile_sum_tensor, profile_count_tensor, device):
    dataset = build_dataset_from_chunk(data, embeddings, profile_sums, profile_counts)
    loader = DataLoader(dataset, batch_size=STREAM_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=(device.type == "cuda"))

    model.eval()
    probs = []
    labels_out = []
    with torch.no_grad():
        for article_idx, customer_idx, pair_counts, visual_similarity, visual_history_count, labels_b in tqdm(loader, desc="Validation", leave=False):
            article_idx = article_idx.to(device, non_blocking=True)
            customer_idx = customer_idx.to(device, non_blocking=True)
            pair_counts = pair_counts.to(device, non_blocking=True)

            article_emb, profiles, visual_similarity_b, history_counts = make_visual_batch(
                article_idx,
                customer_idx,
                pair_counts,
                article_tensor,
                profile_sum_tensor,
                profile_count_tensor,
            )
            logits = model(article_emb, profiles, visual_similarity_b, history_counts)
            probs.append(torch.sigmoid(logits).cpu().numpy())
            labels_out.append(labels_b.numpy())

    y_prob = np.concatenate(probs)
    y_true = np.concatenate(labels_out)
    y_pred = (y_prob >= 0.5).astype("int64")

    del dataset, loader
    gc.collect()
    return {
        "auc_roc": roc_auc_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
    }


## 4. Veriyi Yükleme, Embedding Cache ve Müşteri Profilleri

Bu hücre hazır full embedding cache'i yükler, transaction verisini embedded article evrenine filtreler ve müşteri görsel profil toplamlarını hesaplar. Bu adım late fusion ile aynı görsel evreni kullanır.


In [4]:
set_seed(RANDOM_SEED)
device = get_device("image-history startup")

article_ids, embeddings, article_to_index = load_image_embeddings()
transactions = load_transactions()
transactions, customer_ids, customer_to_index, profile_sums, profile_counts = build_customer_visual_sums(
    transactions,
    embeddings,
    article_to_index,
)

positive_pairs = transactions[["customer_id", "article_id"]].drop_duplicates().reset_index(drop=True)
article_pool = np.array(list(article_to_index.keys()))

log(f"Full positive pairs: {len(positive_pairs):,}")
log(f"Negative article pool: {len(article_pool):,}")


[image-history startup] CUDA available: True | GPU: Tesla T4 | allocated=0.00GB | reserved=0.00GB
Loading embeddings: /kaggle/input/datasets/smoke78/article-image-embeddings-popular-npy/article_image_embeddings_popular.npy
Loading embedding ids: /kaggle/input/datasets/smoke78/article-image-embedding-ids-popular-csv/article_image_embedding_ids_popular.csv
Loaded embeddings: (105100, 1280)
Loading H&M transactions...
Transactions: 31,788,324
Filtering transactions to embedded articles...
Embedded transactions: 31,651,678
Customers with visual history: 1,361,823


Customer visual profiles: 100%|██████████| 31651678/31651678 [01:58<00:00, 267522.99it/s]


Full positive pairs: 27,194,909
Negative article pool: 105,100


## 5. Streaming Image-History Eğitimi

Bu hücre tüm pozitif satın alma çiftlerini streaming chunk yapısıyla gezer. Her pozitif örnek için iki negatif ürün örneklenir. En iyi checkpoint `/kaggle/working/image_history_streaming_full.pt` olarak kaydedilir.


In [5]:
rng = np.random.default_rng(RANDOM_SEED)
validation_positive_count = min(VALIDATION_POSITIVE_ROWS, max(1, len(positive_pairs) // 10))
val_indices = positive_pairs.sample(validation_positive_count, random_state=RANDOM_SEED).index
val_positive = positive_pairs.loc[val_indices].reset_index(drop=True)
train_positive_pairs = positive_pairs.drop(val_indices).reset_index(drop=True)

log(f"Training positive pairs: {len(train_positive_pairs):,}")
log(f"Validation positive pairs: {len(val_positive):,}")

validation_data = prepare_pair_chunk(
    val_positive,
    chunk_id=999_999,
    article_pool=article_pool,
    rng=rng,
    article_to_index=article_to_index,
    customer_to_index=customer_to_index,
)

article_tensor = torch.tensor(embeddings, dtype=torch.float32, device=device)
profile_sum_tensor = torch.tensor(profile_sums, dtype=torch.float32, device=device)
profile_count_tensor = torch.tensor(profile_counts, dtype=torch.float32, device=device)

model = ImageHistoryMLP(image_dim=embeddings.shape[1]).to(device)
pos_weight = torch.tensor([NEGATIVES_PER_POSITIVE], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

best_auc = -1.0
best_state = None
num_chunks = int(np.ceil(len(train_positive_pairs) / STREAM_POSITIVE_CHUNK_SIZE))
log(f"Streaming chunks per epoch: {num_chunks}")

for epoch in range(1, STREAM_EPOCHS + 1):
    log(f"===== Epoch {epoch}/{STREAM_EPOCHS} =====")
    epoch_losses = []
    shuffled = train_positive_pairs.sample(frac=1.0, random_state=RANDOM_SEED + epoch).reset_index(drop=True)

    for chunk_id, start in enumerate(tqdm(range(0, len(shuffled), STREAM_POSITIVE_CHUNK_SIZE), desc=f"Epoch {epoch} chunks")):
        end = min(start + STREAM_POSITIVE_CHUNK_SIZE, len(shuffled))
        positive_chunk = shuffled.iloc[start:end].copy()

        chunk_data = prepare_pair_chunk(
            positive_chunk,
            chunk_id=chunk_id + epoch * 100_000,
            article_pool=article_pool,
            rng=rng,
            article_to_index=article_to_index,
            customer_to_index=customer_to_index,
        )

        loss = train_one_chunk(model, optimizer, criterion, chunk_data, article_tensor, profile_sum_tensor, profile_count_tensor, device)
        epoch_losses.append(loss)
        log(f"Epoch {epoch} | chunk {chunk_id + 1}/{num_chunks} | rows={len(chunk_data):,} | loss={loss:.4f}")

        del positive_chunk, chunk_data
        gc.collect()

    metrics = evaluate_model(model, validation_data, article_tensor, profile_sum_tensor, profile_count_tensor, device)
    log(
        f"Epoch {epoch}/{STREAM_EPOCHS} | "
        f"loss={np.mean(epoch_losses):.4f} | "
        f"val_auc={metrics['auc_roc']:.4f} | "
        f"val_acc={metrics['accuracy']:.4f}"
    )

    if metrics["auc_roc"] > best_auc:
        best_auc = metrics["auc_roc"]
        best_state = {
            "model_state_dict": model.state_dict(),
            "validation_metrics": metrics,
            "config": {
                "stream_positive_chunk_size": STREAM_POSITIVE_CHUNK_SIZE,
                "stream_epochs": STREAM_EPOCHS,
                "stream_batch_size": STREAM_BATCH_SIZE,
                "negatives_per_positive": NEGATIVES_PER_POSITIVE,
                "learning_rate": LEARNING_RATE,
                "full_positive_pairs": len(positive_pairs),
                "validation_positive_rows": len(val_positive),
                "image_dim": embeddings.shape[1],
            },
        }
        torch.save(best_state, MODEL_PATH)
        log(f"Saved best checkpoint: {MODEL_PATH}")

pd.DataFrame([
    {
        "model": "image_history_streaming_full",
        "auc_roc": best_state["validation_metrics"]["auc_roc"],
        "accuracy": best_state["validation_metrics"]["accuracy"],
        "full_positive_pairs": len(positive_pairs),
        "validation_positive_rows": len(val_positive),
        "image_dim": embeddings.shape[1],
        "checkpoint": str(MODEL_PATH),
    }
]).to_csv(RESULTS_PATH, index=False)

log(f"Image-history training complete. Best validation AUC: {best_auc:.4f}")
log(f"Saved results: {RESULTS_PATH}")


Training positive pairs: 27,074,909
Validation positive pairs: 120,000
Streaming chunks per epoch: 136
===== Epoch 1/3 =====


Visual feature batches: 100%|██████████| 12/12 [00:11<00:00,  1.05it/s]
                                                                       

Epoch 1 | chunk 1/136 | rows=599,994 | loss=0.7360


Visual feature batches: 100%|██████████| 12/12 [00:11<00:00,  1.06it/s]
                                                                       

Epoch 1 | chunk 2/136 | rows=599,997 | loss=0.6916


...

Epoch 3 | chunk 135/136 | rows=599,991 | loss=0.4923


Visual feature batches: 100%|██████████| 5/5 [00:04<00:00,  1.24it/s]
                                                                     

Epoch 3 | chunk 136/136 | rows=224,726 | loss=0.4908


Epoch 3 chunks: 100%|██████████| 136/136 [1:15:29<00:00, 33.30s/it]


Epoch 3/3 | loss=0.4950 | val_auc=0.9237 | val_acc=0.8297
Saved best checkpoint: /kaggle/working/image_history_streaming_full.pt
Image-history training complete. Best validation AUC: 0.9237
Saved results: /kaggle/working/image_history_full_results.csv
